# Demo 5 — Standalone pairwise correlation analysis

When several variables all trend together, raw correlations cannot separate direct
association from shared drift — partial correlations can. This demo shows that on
the `longley` dataset (R base; Longley 1967), a classic econometrics benchmark
built to be severely multicollinear: 16 annual US macroeconomic observations
(1947–1962) in which GNP, population, and the calendar year all move together.

No model is fitted — the analysis is purely exploratory. Alongside ordinary
Pearson correlations it computes partial correlations (the association between two
variables after regressing out all the others), separating direct relationships
from shared trends. The `constraints = 'Year > 1950'` filter restricts the
analysis to the post-war growth period, illustrating row filtering.

## Run on Google Colab

On [Google Colab](https://colab.research.google.com)? Run the cell below first —
it installs kbstatpy, its R packages, and the demo data (~3–5 min the first time).
It is a no-op when you run this notebook locally from the kbstatpy source tree.
Then run the cells below to see the tables and figures rendered inline.

In [ ]:
# Google Colab only: install kbstatpy + its R packages + the demo data.
# (Does nothing when the notebook runs locally from the source tree.)
import sys
if 'google.colab' in sys.modules:
    !curl -sSL https://raw.githubusercontent.com/kimbostroem/kbstatpy/master/demos/colab_setup.sh | bash

## Setup

In [ ]:
import os

import matplotlib.pyplot as plt
from kbstatpy import Kbstat, KbstatOptions

## Options

No `y` or `x` — this is a pure correlation run. `constraints` filters to years after 1950. No model is fitted.

In [ ]:
options = KbstatOptions()
options.in_file     = os.path.join(options.demo_dir, 'data/longley.csv')
options.out_dir  = ''   # empty: show results inline only; set a folder to also save them
options.correlation = 'GNP.deflator, GNP, Unemployed, Population, Year'
options.constraints = 'Year > 1950'
options.rename      = 'GNP.deflator -> GNP_Deflator; Unemployed -> Unemployment'

## Run

`run()` detects that only `correlation` is set and calls `correlate()` directly.

Files are written only if `out_dir` is set (empty here, so results are shown inline).

In [ ]:
kb = Kbstat(options)
kb.run();

## Save results

Everything above is shown inline. To also write all tables and figures to disk, uncomment the two lines below and run this cell — it sets `out_dir` and calls `save()` to write the results already computed above (no re-run).

In [ ]:
# options.out_dir = 'results/demo_05_correlation'
# kb.save()
# kb.download_link()   # remote server: zip the results and click to download

## Interpretation

- The Longley dataset is a classic high-multicollinearity benchmark — correlations close to 1.0 are expected.
- All macroeconomic variables trend together over time, so near-perfect correlations reflect shared temporal trends rather than direct causal links.
- The scatter plot grid shows the pairwise relationships visually.

### Reading the two tables together

The raw and partial tables say most when compared, because it is the *difference* between them that carries the message.

- A **high raw correlation that collapses in the partial** means the pair is largely explained by the other variables. That is exactly what Longley is built to show: GNP, population and the calendar year all rise together, so almost any two of them correlate near 1.0 while little survives once the rest are held fixed. A whole block behaving this way is a sign that those variables track one underlying quantity rather than several distinct ones.
- A **partial that stays high** means the pair shares something the other variables do not capture — the association worth a second look, since it survives everything else in the set.
- A **low raw correlation that grows in the partial** means the other variables were masking, i.e. suppressing, the association.

### What conditioning can and cannot tell you

Partial correlation removes whatever is *linearly predictable* from the conditioning set. It has no notion of cause, so the same arithmetic does three very different things depending on what those variables actually are:

| the conditioned variable is a | effect on the reported partial |
|---|---|
| **common cause** (confounder) of X and Y | removes a spurious association — the intended use |
| **common effect** (collider), i.e. X and Y both influence it | **creates** an association between variables that are in fact unrelated |
| **mediator** on a path X → M → Y | removes a real effect, leaving only the direct part |

Longley is the benign case: calendar time plausibly drives every series, so conditioning on the others largely removes shared drift. But note that the confounder and mediator rows produce the *same* signature — high raw, near-zero partial — with opposite meanings: in one the association was never real, in the other it is real and has just been conditioned away. Nothing in the data distinguishes them, only subject knowledge does. Since the conditioning set here is simply *all remaining variables*, read the partials as a statement about redundancy within the variable set rather than as evidence about mechanism.